In [9]:
import sys
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *

# ============================================
# FIXED DATA CLEANING FUNCTIONS
# ============================================

def clean_any_data(path, sheet_name=None, **kwargs):
    """
    Universal data cleaner with proper error handling
    """
    print(f"\n📁 READING FILE: {os.path.basename(path)}")
    
    file_ext = Path(path).suffix.lower()
    df = None
    
    # Try different encodings and delimiters
    encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1', 'utf-16', 'cp1250']
    
    if file_ext in ['.xlsx', '.xls', '.xlsm']:
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, engine='openpyxl')
            print("✅ Read as Excel file")
        except:
            try:
                df = pd.read_excel(path, sheet_name=sheet_name, engine='xlrd')
                print("✅ Read as Excel file (xlrd engine)")
            except Exception as e:
                print(f"⚠️ Excel read failed: {e}")
                return None
    
    elif file_ext in ['.csv', '.txt', '.dat', '.tsv']:
        delimiters = [';', ',', '\t', '|', ' ']
        df = None
        
        for encoding in encodings:
            for delim in delimiters:
                try:
                    df = pd.read_csv(
                        path, 
                        sep=delim, 
                        encoding=encoding,
                        engine='python',
                        on_bad_lines='skip',
                        **kwargs
                    )
                    # Check if we got meaningful data
                    if df is not None and not df.empty and df.shape[1] > 1:
                        print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                        break
                except:
                    continue
            if df is not None and not df.empty and df.shape[1] > 1:
                break
        
        # If still None, try reading without delimiter detection
        if df is None or df.empty:
            print("⚠️ Auto-detection failed, trying manual parsing...")
            df = manual_parse_text_file(path)
    
    else:
        print(f"⚠️ Unknown file type: {file_ext}, trying to read as text...")
        try:
            df = manual_parse_text_file(path)
        except:
            return None
    
    # Check if df is valid
    if df is None:
        print("❌ Failed to read file")
        return None
    
    if df.empty:
        print("❌ File is empty")
        return None
    
    print(f"📊 Raw data shape: {df.shape}")
    
    # Clean the data
    try:
        # Handle single column data
        df = split_single_column(df)
        
        # Clean column names
        df = clean_column_names(df)
        
        # Remove header rows if present
        df = remove_header_rows(df)
        
        # Standardize missing values
        df = standardize_missing_values(df)
        
        # Clean numeric columns
        df = clean_numeric_columns(df)
        
        # Clean text columns
        df = clean_text_columns(df)
        
        # Remove empty rows and columns
        df = remove_empty(df)
        
        print(f"✅ Final shape: {df.shape}")
        return df
        
    except Exception as e:
        print(f"❌ Cleaning error: {e}")
        return df  # Return the partially cleaned data

def manual_parse_text_file(path):
    """Manually parse text files when auto-detection fails"""
    try:
        with open(path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    # Try to detect delimiter
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        
        df = pd.DataFrame(data)
        
        # Check if first row is header
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    
    return None

def split_single_column(df):
    """Split single-column dataframes"""
    if df is None or df.empty:
        return df
        
    if df.shape[1] == 1:
        col_name = df.columns[0]
        col_data = df[col_name].astype(str)
        sample = col_data.iloc[0] if len(col_data) > 0 else ''
        delimiters = [';', ',', '\t', '|']
        
        for delim in delimiters:
            if delim in sample:
                split_data = col_data.str.split(delim, expand=True)
                try:
                    pd.to_numeric(split_data.iloc[0, 0])
                    split_data.columns = [f'col_{i+1}' for i in range(split_data.shape[1])]
                except:
                    headers = split_data.iloc[0].tolist()
                    split_data = split_data.iloc[1:].copy()
                    headers = [str(h).strip() for h in headers]
                    split_data.columns = headers
                return split_data.reset_index(drop=True)
    return df

def clean_column_names(df):
    """Clean column names"""
    if df is None or df.empty:
        return df
        
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.loc[:, ~df.columns.astype(str).str.contains(r',+$', na=False)]
    df.columns = [f'col_{i}' if str(col).strip() == '' or pd.isna(col) else col for i, col in enumerate(df.columns)]
    df.columns = [re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col)).strip('_') for col in df.columns]
    
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

def remove_header_rows(df):
    """Remove rows that look like headers"""
    if df is None or df.empty:
        return df
        
    if len(df) > 0:
        first_row = df.iloc[0].astype(str)
        header_keywords = ['sample', 'point', 'id', 'name', 'date', 'time', 
                          'value', 'measurement', 'parameter', 'unit']
        header_score = 0
        for val in first_row:
            if any(keyword in str(val).lower() for keyword in header_keywords):
                header_score += 1
        
        if header_score / len(first_row) > 0.3:
            df = df.iloc[1:].reset_index(drop=True)
    return df

def standardize_missing_values(df):
    """Standardize missing values"""
    if df is None or df.empty:
        return df
        
    missing_values = ['', 'nan', 'NaN', 'NAN', 'null', 'NULL', 'None', 
                     'na', 'NA', '-', '--', '---', 'n/a', 'N/A', '?']
    df = df.replace(missing_values, np.nan)
    
    for col in df.columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'^<.*$', 'nan', regex=True)
            df[col] = df[col].astype(str).str.replace(r'^>.*$', 'nan', regex=True)
    return df

def clean_numeric_columns(df):
    """Clean numeric columns"""
    if df is None or df.empty:
        return df
        
    text_cols = ['sample', 'point_id', 'point', 'id', 'name', 
                 'lithology', 'description', 'comment', 'remarks',
                 'unit', 'method', 'date', 'time']
    
    def clean_numeric(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().replace(',', '.')
        if x.startswith('<') or x.startswith('>'):
            return np.nan
        if x in ['', 'nan', 'NaN', 'NAN', 'null', 'NULL']:
            return np.nan
        # Remove any non-numeric characters except decimal point and minus sign
        x = re.sub(r'[^\d.\-eE+]', '', x)
        try:
            return float(x)
        except:
            return np.nan
    
    for col in df.columns:
        col_lower = col.lower()
        is_text = any(text in col_lower for text in text_cols)
        if not is_text:
            try:
                df[col] = df[col].map(clean_numeric)
            except:
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
    return df

def clean_text_columns(df):
    """Clean text columns"""
    if df is None or df.empty:
        return df
        
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('', np.nan)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
    return df

def remove_empty(df):
    """Remove empty rows and columns"""
    if df is None or df.empty:
        return df
        
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    return df

# ============================================
# MAIN APPLICATION - FIXED VERSION
# ============================================

class GeochemicalCleaner(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.current_file = None
        self.init_ui()
        
    def init_ui(self):
        """Initialize the user interface"""
        self.setWindowTitle("🧪 Geochemical Data Cleaner Pro")
        self.setGeometry(100, 100, 1000, 700)
        
        # Central widget
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QVBoxLayout()
        central_widget.setLayout(main_layout)
        
        # Toolbar
        toolbar = self.addToolBar("Main")
        
        upload_action = QAction("📁 Upload Data", self)
        upload_action.triggered.connect(self.upload_file)
        toolbar.addAction(upload_action)
        
        clean_action = QAction("🧹 Clean Data", self)
        clean_action.triggered.connect(self.clean_data)
        toolbar.addAction(clean_action)
        
        export_action = QAction("💾 Export", self)
        export_action.triggered.connect(self.export_data)
        toolbar.addAction(export_action)
        
        # Info label
        self.info_label = QLabel("No data loaded. Click 'Upload Data' to begin.")
        self.info_label.setStyleSheet("font-size: 14px; font-weight: bold; padding: 10px; background-color: #f0f0f0; border-radius: 5px;")
        main_layout.addWidget(self.info_label)
        
        # Table
        self.table = QTableWidget()
        self.table.setAlternatingRowColors(True)
        main_layout.addWidget(self.table)
        
        # Status bar
        self.statusBar = QStatusBar()
        self.setStatusBar(self.statusBar)
        self.statusBar.showMessage("Ready")
        
        # Apply styles
        self.setStyleSheet("""
            QMainWindow { background-color: #f5f5f5; }
            QTableWidget { background-color: white; gridline-color: #d0d0d0; }
            QTableWidget::item:selected { background-color: #4CAF50; color: white; }
            QPushButton { background-color: #2196F3; color: white; padding: 8px 15px; border-radius: 4px; border: none; }
            QPushButton:hover { background-color: #1976D2; }
        """)
        
    def upload_file(self):
        """Upload data file"""
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Open Geochemical Data",
            "",
            "All Files (*);;CSV Files (*.csv);;Excel Files (*.xlsx *.xls);;Text Files (*.txt)"
        )
        
        if file_path:
            self.current_file = file_path
            self.statusBar.showMessage(f"Loading: {os.path.basename(file_path)}...")
            
            try:
                # Use the fixed cleaning function
                self.df = clean_any_data(file_path)
                
                if self.df is not None and not self.df.empty:
                    self.display_data()
                    self.statusBar.showMessage(f"✅ Loaded: {os.path.basename(file_path)} - {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                    self.info_label.setText(f"📊 Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns")
                else:
                    QMessageBox.warning(self, "Error", "No data could be read from the file.")
                    self.statusBar.showMessage("❌ Error loading file")
                    
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Failed to load file:\n{str(e)}")
                self.statusBar.showMessage("❌ Error loading file")
                import traceback
                traceback.print_exc()
                
    def clean_data(self):
        """Clean the loaded data"""
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
            
        self.statusBar.showMessage("Cleaning data...")
        
        try:
            # Apply additional cleaning
            self.cleaned_df = self.df.copy()
            
            # Convert all columns to numeric if possible
            for col in self.cleaned_df.columns:
                try:
                    self.cleaned_df[col] = pd.to_numeric(self.cleaned_df[col], errors='coerce')
                except:
                    pass
            
            # Remove rows with all NaN
            self.cleaned_df = self.cleaned_df.dropna(how='all')
            
            # Remove columns with all NaN
            self.cleaned_df = self.cleaned_df.dropna(axis=1, how='all')
            
            self.display_data(self.cleaned_df)
            self.statusBar.showMessage(f"✅ Data cleaned! {self.cleaned_df.shape[0]} rows, {self.cleaned_df.shape[1]} columns")
            QMessageBox.information(self, "Success", "Data cleaning completed successfully!")
            
        except Exception as e:
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
            self.statusBar.showMessage("❌ Error cleaning data")
            import traceback
            traceback.print_exc()
            
    def display_data(self, df=None):
        """Display data in table"""
        if df is None:
            if self.cleaned_df is not None:
                df = self.cleaned_df
            elif self.df is not None:
                df = self.df
            else:
                return
                
        self.table.setRowCount(min(len(df), 1000))
        self.table.setColumnCount(len(df.columns))
        self.table.setHorizontalHeaderLabels(df.columns.astype(str))
        
        # Fill data
        for i in range(min(len(df), 1000)):
            for j, col in enumerate(df.columns):
                val = df.iloc[i, j]
                if pd.isna(val):
                    self.table.setItem(i, j, QTableWidgetItem(""))
                else:
                    self.table.setItem(i, j, QTableWidgetItem(str(val)))
                    
        # Resize columns
        self.table.resizeColumnsToContents()
        
    def export_data(self):
        """Export cleaned data"""
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "Warning", "No data to export!")
            return
            
        df_to_export = self.cleaned_df if self.cleaned_df is not None else self.df
        
        file_path, _ = QFileDialog.getSaveFileName(
            self,
            "Save Cleaned Data",
            "cleaned_data.csv",
            "CSV Files (*.csv);;Excel Files (*.xlsx);;All Files (*)"
        )
        
        if file_path:
            try:
                if file_path.endswith('.csv'):
                    df_to_export.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif file_path.endswith('.xlsx'):
                    df_to_export.to_excel(file_path, index=False)
                else:
                    df_to_export.to_csv(file_path + '.csv', sep=';', index=False, encoding='utf-8-sig')
                    
                QMessageBox.information(self, "Success", f"Data exported to {file_path}")
                self.statusBar.showMessage(f"✅ Exported: {file_path}")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")


# ============================================
# RUN THE APPLICATION
# ============================================

def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    
    # Set application font
    font = QFont("Segoe UI", 10)
    app.setFont(font)
    
    window = GeochemicalCleaner()
    window.show()
    
    sys.exit(app.exec_())


if __name__ == '__main__':
    main()

SystemExit: 0

C:\Users\HP\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
# In Jupyter cell
%run -i geochemical_cleaner.py
# Or
from geochemical_cleaner import launch_app
app, window = launch_app()

Exception: File `'geochemical_cleaner.py'` not found.

In [7]:
import pandas as pd
import io

# Your data as text
data = """Sample code	CuPPM	PbPPM	ZnPPM	Au_ppm	Ag_PPM	Ba_PPM	As_ppm	Total_Concentration
WR2-1	0.000000057	0.44798	1	2.606E-09	0.000000864	19	15.7	1.447980924
WR2-2	0.000000685	0.60522	1	5.8095E-08	0.000001	16	90.1	1.605221743
WR2-3	0.000001491	0.27309	1	5.801E-09	0.000000471	5	46.4	1.273091968
WR2-4	0.0001	1	1	2.9948E-08	0.000001	249	117	2.00010103"""

# Read the data
df = pd.read_csv(io.StringIO(data), sep='\t')
print("Original data:")
print(df)
print("\nData types:")
print(df.dtypes)

# Clean numeric columns
for col in df.columns:
    if col != 'Sample code':
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("\nCleaned data:")
print(df)
print("\nCleaned data types:")
print(df.dtypes)

# Save to CSV
df.to_csv('cleaned_data.csv', index=False)
print("\n✅ Saved to cleaned_data.csv")

Original data:
  Sample code         CuPPM    PbPPM  ZnPPM        Au_ppm        Ag_PPM  \
0       WR2-1  5.700000e-08  0.44798      1  2.606000e-09  8.640000e-07   
1       WR2-2  6.850000e-07  0.60522      1  5.809500e-08  1.000000e-06   
2       WR2-3  1.491000e-06  0.27309      1  5.801000e-09  4.710000e-07   
3       WR2-4  1.000000e-04  1.00000      1  2.994800e-08  1.000000e-06   

   Ba_PPM  As_ppm  Total_Concentration  
0      19    15.7             1.447981  
1      16    90.1             1.605222  
2       5    46.4             1.273092  
3     249   117.0             2.000101  

Data types:
Sample code             object
CuPPM                  float64
PbPPM                  float64
ZnPPM                    int64
Au_ppm                 float64
Ag_PPM                 float64
Ba_PPM                   int64
As_ppm                 float64
Total_Concentration    float64
dtype: object

Cleaned data:
  Sample code         CuPPM    PbPPM  ZnPPM        Au_ppm        Ag_PPM  \
0       

In [ ]:
cd C:\Users\HP
python geochemical_cleaner.py